# Exportação de modelos 3D experimentais

Este notebook extrai o crânio ósseo da série CT `skull`, usando um limiar calibrado em HU para excluir tecidos moles e preservar as regiões frontal e basal. A segmentação é exploratória e deve ser conferida por um profissional antes de qualquer uso clínico.

Os arquivos PLY e STL são criados em `results/models/`. As coordenadas usam o espaçamento físico DICOM; a orientação absoluta do paciente não é aplicada.

O notebook preserva arquivos existentes com o mesmo nome e os verifica antes de reutilizá-los.

## 1. Carregar a série e preparar os parâmetros

Carregamos a série DICOM `skull`, conferimos a calibração HU e definimos o limiar ósseo em 300 HU. A máscara considera todas as fatias e mantém o maior componente ósseo conectado, sem recortar a região frontal.

In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv
import pydicom
import sys
from scipy import ndimage
from skimage.measure import marching_cubes

current_dir = Path.cwd().resolve()
search_dirs = [current_dir, *current_dir.parents]
search_dirs.extend(parent / "reconstrucao-cranio" for parent in search_dirs.copy())
project_root = next(
    (folder for folder in search_dirs if (folder / "data" / "skull").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Não encontrei data/skull. "
        f"Diretório de trabalho atual do Jupyter: {current_dir}."
    )

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from leitura import ler_dicom_volume

data_dir = project_root / "data" / "skull" / "0051" / "0051"
model_dir = project_root / "results" / "models"
volume, voxel_spacing_zyx, image_files = ler_dicom_volume(data_dir)
if volume.ndim != 3 or volume.shape[0] != len(image_files):
    raise ValueError(
        f"Volume DICOM inesperado: dimensões {volume.shape}, tipo {volume.dtype}."
    )

calibracoes = {
    (
        float(dataset.get("RescaleSlope", 1)),
        float(dataset.get("RescaleIntercept", 0)),
    )
    for dataset in (pydicom.dcmread(path, stop_before_pixels=True) for path in image_files)
}
if len(calibracoes) != 1:
    raise ValueError(f"A calibração HU varia entre os cortes DICOM: {calibracoes}")
rescale_slope, rescale_intercept = calibracoes.pop()
if (
    not np.isfinite(rescale_slope)
    or rescale_slope <= 0
    or not np.isfinite(rescale_intercept)
):
    raise ValueError("A calibração HU informada pela série DICOM é inválida.")

bone_threshold_hu = 300.0
bone_threshold_pixels = (bone_threshold_hu - rescale_intercept) / rescale_slope
filename_label = f"bone_hu{int(bone_threshold_hu)}_full"
step_size = 2
print(f"Volume CT: {volume.shape}, tipo {volume.dtype}")
print(f"Espaçamento DICOM (z, y, x), em mm: {voxel_spacing_zyx}")
print(f"Limiar ósseo: {bone_threshold_hu:g} HU ({bone_threshold_pixels:g} pixels DICOM)")
print("Recorte anatômico: nenhum; tecidos moles excluídos pelo limiar ósseo")
print(f"Pasta de saída: {model_dir}")

## 2. Isolar o crânio e gerar a malha

O limiar de 300 HU exclui tecidos moles sem cortar a região frontal ou a base do crânio. Conservamos o maior componente ósseo conectado para remover estruturas externas. Marching Cubes recebe o espaçamento na ordem `(z, y, x)`; depois reorganizamos os vértices para `(x, y, z)`, ordem usada por PyVista.

In [ ]:
bone_mask = volume >= bone_threshold_pixels
component_labels, component_count = ndimage.label(bone_mask)
if component_count == 0:
    raise ValueError("O limiar ósseo não encontrou componentes na região do crânio.")
component_sizes = np.bincount(component_labels.ravel())
largest_component = int(np.argmax(component_sizes[1:]) + 1)
bone_mask = component_labels == largest_component
if not bone_mask.any():
    raise ValueError("A máscara do maior componente ósseo está vazia.")

mask = bone_mask
vertices_zyx, triangles, _, _ = marching_cubes(
    mask.astype(np.uint8),
    level=0.5,
    spacing=voxel_spacing_zyx,
    step_size=step_size,
    allow_degenerate=False,
)
vertices_xyz = vertices_zyx[:, [2, 1, 0]]
faces = np.column_stack(
    (np.full(len(triangles), 3, dtype=np.int64), triangles)
).ravel()
mesh = pv.PolyData(vertices_xyz, faces)
meshes = {filename_label: mesh}
model_dir.mkdir(parents=True, exist_ok=True)
print(
    f"Crânio ósseo completo: {mesh.n_points:,} vértices, {mesh.n_cells:,} triângulos, "
    f"componente {int(component_sizes[largest_component]):,} voxels; "
    f"outros componentes ignorados: {component_count - 1}"
)

## 3. Exportar PLY e STL e reler para conferir

Gravamos cada malha nos dois formatos. Imediatamente depois, reabrimos cada arquivo com PyVista e comparamos a quantidade de células e os limites das coordenadas com a malha em memória. A releitura confirma que o arquivo contém uma geometria e preserva seus limites; não demonstra que o limiar ou a anatomia estejam corretos.

In [ ]:
export_summary = []

for label, mesh in meshes.items():
    for extension in ("ply", "stl"):
        output_path = model_dir / f"skull_{label}_step{step_size}.{extension}"
        if output_path.exists():
            existing_mesh = pv.read(output_path)
            if existing_mesh.n_points == 0 or existing_mesh.n_cells == 0:
                print(f"Arquivo existente inválido; regenerando: {output_path.name}")
                mesh.save(output_path)
            elif existing_mesh.n_cells != mesh.n_cells or not np.allclose(existing_mesh.bounds, mesh.bounds):
                print(f"Arquivo existente não corresponde à malha atual; regenerando: {output_path.name}")
                mesh.save(output_path)
            else:
                print(f"Arquivo existente preservado e verificado: {output_path.name}")
        else:
            mesh.save(output_path)
        reloaded_mesh = pv.read(output_path)

        if reloaded_mesh.n_points == 0 or reloaded_mesh.n_cells == 0:
            raise ValueError(f"A malha exportada está vazia: {output_path}")
        if reloaded_mesh.n_cells != mesh.n_cells:
            raise ValueError(f"A quantidade de faces mudou após exportar: {output_path}")
        if not np.allclose(reloaded_mesh.bounds, mesh.bounds):
            raise ValueError(f"As coordenadas mudaram após exportar: {output_path}")

        export_summary.append(
            {
                "file": output_path,
                "threshold_hu": bone_threshold_hu,
                "format": extension,
                "size_bytes": output_path.stat().st_size,
                "triangles": reloaded_mesh.n_cells,
                "bounds_xyz": reloaded_mesh.bounds,
            }
        )

for item in export_summary:
    print(
        f"{item['file'].name}: {item['format'].upper()}, "
        f"{item['triangles']:,} triângulos, "
        f"{item['size_bytes']:,} bytes, "
        f"limites XYZ={item['bounds_xyz']}"
    )

if len(export_summary) != 2:
    raise ValueError(f"Esperados 2 arquivos conferidos; obtidos {len(export_summary)}.")

## 4. Visualizar um arquivo exportado

Reabrimos do disco a superfície óssea exportada em formato PLY para conferir o resultado.

In [ ]:
preview_path = model_dir / f"skull_{filename_label}_step{step_size}.ply"
preview_mesh = pv.read(preview_path)
plotter = pv.Plotter(notebook=True, window_size=(900, 600))
plotter.add_text(f"Crânio ósseo completo — limiar {bone_threshold_hu:g} HU", font_size=12)
plotter.add_mesh(preview_mesh, color="wheat", smooth_shading=True)
plotter.show_axes()
plotter.camera_position = "iso"
plotter.show(jupyter_backend="trame")

## Resultado e limitações

Os arquivos em `results/models/` contêm o maior componente ósseo conectado acima do limiar escolhido, sem recortes espaciais. Tecidos moles são excluídos, mas ossos faciais que estejam conectados ao crânio permanecem. Confira o modelo em diferentes ângulos.

As coordenadas usam o espaçamento físico DICOM, mas não incluem a orientação absoluta do paciente. Um STL não é automaticamente uma malha fechada ou apta para impressão 3D; exportar corretamente não comprova acurácia anatômica.